In [1]:
# Cell 1
# Mục đích: định nghĩa lại client và các hàm đo byte (notebook mới phải khai báo lại).
from google.cloud import bigquery

PROJECT = "bq-learning-510104"
client = bigquery.Client(project=PROJECT, location="US")   # kênh kết nối tới BigQuery (vùng US)

def mb(n):
    # đổi byte sang MB; trả None nếu n là None để lỗi "đọc byte quá sớm" lộ ra
    return None if n is None else round(n / 1024 / 1024, 2)

def check(sql):
    # dry run: chỉ ước lượng số MB sẽ quét, không chạy thật
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    job = client.query(sql, job_config=cfg)
    print(f"Se quet: {mb(job.total_bytes_processed)} MB")

def stats(sql, max_mb=100):
    # chạy thật, có trần byte; in processed và billed; trả về DataFrame
    cfg = bigquery.QueryJobConfig(use_query_cache=False,
                                  maximum_bytes_billed=max_mb * 1024 * 1024)
    job = client.query(sql, job_config=cfg)
    df = job.result().to_dataframe()          # result() chờ job xong rồi mới đọc byte
    print(f"processed: {mb(job.total_bytes_processed)} MB | billed: {mb(job.total_bytes_billed)} MB")
    return df

In [2]:
# Cell 2
# Mục đích: liệt kê các query dbt đã gửi trong 6 giờ qua và xem mỗi query bị tính bao nhiêu byte.
sql = f"""
SELECT
  creation_time,
  total_bytes_processed,
  total_bytes_billed,
  REGEXP_EXTRACT(query, r'"node_id": "([^"]+)"') AS node_id   -- tên model/test, lấy từ chú thích dbt gắn đầu query
FROM `{PROJECT}.region-us.INFORMATION_SCHEMA.JOBS`
WHERE creation_time >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 6 HOUR)
  AND query LIKE '%"app": "dbt"%'
  AND query NOT LIKE '%INFORMATION_SCHEMA%'
ORDER BY creation_time
"""
check(sql)                       # xem trước số MB sẽ quét
df_j = stats(sql, max_mb=100)    # chạy thật, có trần 100 MB
print("Số dòng:", len(df_j))     # in trước khi lặp: bảng rỗng thì vòng for im lặng chạy 0 lần

total = 0
for _, row in df_j.iterrows():   # _ nhận số thứ tự, row nhận cả một dòng
    print(row["creation_time"], "|", mb(row["total_bytes_billed"]), "MB |", row["node_id"])
    total += row["total_bytes_billed"]
print("-" * 60)
print("Tổng billed:", mb(total), "MB")

Se quet: 0.03 MB
processed: 0.01 MB | billed: 20.0 MB
Số dòng: 15
2026-10-13 04:53:01.664000+00:00 | 10.0 MB | test.bi_dbt.unique_fct_net_sales_order_item_id.69c2ff268d
2026-10-13 04:53:01.665000+00:00 | 20.0 MB | test.bi_dbt.relationships_fct_net_sales_product_id__product_id__ref_dim_products_.f594ea4f41
2026-10-13 04:53:01.669000+00:00 | 10.0 MB | test.bi_dbt.not_null_fct_net_sales_order_item_id.c5c9bd5463
2026-10-13 04:53:01.705000+00:00 | 10.0 MB | test.bi_dbt.not_null_fct_net_sales_product_id.7886f7c168
2026-10-13 05:03:44.727000+00:00 | 10.0 MB | test.bi_dbt.unique_dim_products_product_id.ef7b2a9985
2026-10-13 05:03:44.731000+00:00 | 10.0 MB | test.bi_dbt.not_null_dim_products_product_id.c8aba288d1
2026-10-13 05:03:44.776000+00:00 | 0.0 MB | test.bi_dbt.relationships_fct_net_sales_product_id__product_id__ref_dim_products_.f594ea4f41
2026-10-13 05:28:39.402000+00:00 | 0.0 MB | test.bi_dbt.relationships_fct_net_sales_product_id__product_id__ref_dim_products_.f594ea4f41
2026-10-13 0